# Find scatterplot structure within groups

Select `detectors=["partition_scagnostics"]` in `onceover.profile()` to discover groups that reveal stronger scatterplot structure. The detector is available on request. It returns `partitioned_shape` findings in the ordinary profile report. Use `onceover.partition_scagnostics(df, ...)` to specify the pairs, candidate context, or metric explicitly.

In this example the table mixes two protocols. The same `x` values appear under both. In protocol A, `y` increases as `x` increases. In protocol B, it decreases. Pass `protocol` as a candidate column and let onceover choose the rule.

In [1]:
import onceover
import polars as pl

x = list(range(-100, 101))
df = pl.DataFrame({
    "x": x + x,
    "y": x + [-value for value in x],
    "protocol": ["A"] * len(x) + ["B"] * len(x),
})

report = onceover.profile(df, detectors=["partition_scagnostics"])
print(report.select(type="partitioned_shape")[0]["claim"])

# Use the scoped call when you want to choose the pair, context, and metric.
result = onceover.partition_scagnostics(
    df,
    pairs=[("x", "y")],
    partition_by=["protocol"],
    metrics=["monotonic"],
)

finding = result["findings"][0]
held = finding["validation"]
print("Rule", finding["rule"])
print("Pooled ρ", round(held["pooled"]["rho"], 3))
print("Group ρ", [round(g["shape"]["rho"], 3) for g in held["groups"]])
print("Rows per group", [g["rows"] for g in held["groups"]])

x × y is more monotonic across protocol = "A" versus the remainder. Held-out Spearman ρ is +0.13 pooled; within groups: A +1.00, B -1.00.
Rule {'kind': 'levels', 'column': 'protocol', 'levels': ['A']}
Pooled ρ 0.13
Group ρ [1.0, -1.0]
Rows per group [110, 91]


## Choose what to search

- `pairs=[("x", "y")]` selects the plotted numeric columns. Omit it to consider up to 16 numeric pairs in column-name order. Explicit pairs are visited in the order you give them.
- `partition_by=["protocol", "site"]` names candidate explanations. Strings, booleans, categoricals, and enums supply groups. Numeric candidates supply quantile cutoffs. Cast numeric group labels to strings to use categorical rules. Omit this argument to consider supported context columns up to the search caps. The search excludes either axis as context for its own pair.
- `metrics=["monotonic"]` selects squared Spearman. `"binned_striated"` measures alignment in a grid-based minimum spanning tree. The default evaluates both.
- `seed=0` is the default. It controls sample selection, the discovery and validation split, and label permutations for a fixed input row order.

## Read the rule and coverage

A `levels` rule places each listed level in its own group, in list order. The final group contains every remaining row. Here, group zero is protocol A and group one is the remainder. A `threshold` rule places finite values at or below the cutoff in group zero and the remainder in group one. A `cross` combines two equality tests from binary columns. Its group index is `2 * a + b`, where each test is zero on a match and one otherwise.

The `discovery` and `validation` records both carry the pooled score, the group scores and sizes, and their weighted gain. Read `validation` and the finding's `adjusted_p_value` first. For `monotonic`, signed `rho` shows the direction and `score` is its square. `result["coverage"]` shows the 4,096-row sampling cap, examined pairs and rules, unsupported partitions, and validation outcomes.

Read the [methodology](../methodology/partition_scagnostics.ipynb) for the objective, null model, and work bounds, and the [case studies](../case_studies/partition_scagnostics_cases.ipynb) for real and synthetic examples.